# Data Visualizations

In [ ]:
# import packages 
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import glob
import os

In [2]:
# colors 

'#f5f0e8'   # body
'#1a1a1a'   # background
'#06395b'
'#0c629b'
"#458dbd"
"#adddfc"
'#1a5c3a'
"#22794c"
"#67a384"
"#b4d9c6"
"#c48b22"
"#e3b96b"
"#f7dfb3"
"#b84141"

color1 = ['#06395b', "#458dbd", "#adddfc", 'rgba(6, 57, 91, 0.3)', 'rgba(69, 141, 189, 0.3)' ]
color2 = ['#1a5c3a', "#4f916f", "#b4d9c6", 'rgba(79, 145, 111, 0.3)', 'rgba(180, 217, 198, 0.3)' ]
color3 = ["#c48b22", "#e3b96b", "#f7dfb3", 'rgba(196, 139, 34, 0.3)', 'rgba(227, 185, 107, 0.3)' ]

## Overall Percent Spending used on Food

Food personal consumption expenditures / total personal consumption expenditures over time = what percent of spending is used for food over time in general


In [3]:
# import data

food_pce = pd.read_csv("../data/total/DFXARC1M027SBEA.csv", low_memory=False)
pce = pd.read_csv("../data/total/PCE.csv", low_memory=False)

In [4]:
# clean data 

# filter to 1960 and later 
food_pce_clean = food_pce.loc[food_pce['observation_date']>='1960-01-01']
pce_clean = pce.loc[pce['observation_date']>='1960-01-01']

# rename 
food_pce_clean = food_pce_clean.rename(columns={'DFXARC1M027SBEA':'food_pce'})
pce_clean = pce_clean.rename(columns={'PCE':'pce'})

# merge 
percent_food_expend = food_pce_clean.merge(pce_clean, how='outer')

# divide to get percentage
percent_food_expend['percent'] = (percent_food_expend['food_pce']/percent_food_expend['pce']) *100

In [5]:
# line chart of food as a percent of personal consumption expenditures

# data 
y = percent_food_expend["percent"]
x = percent_food_expend["observation_date"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=y,
        mode="lines",
        name="",
        line=dict(
            color='#67a384',
            width=4
        ),
        fill='tozeroy',
        fillcolor='rgba(103, 163, 132, 0.18)',
        hovertemplate=
        "Food Percentage: %{y:.1f}%<extra></extra>"
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Food Expenditure as a Share of Total Personal Consumption Expenditures",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=10
    ),

    yaxis=dict(
        title="Percentage",
        range=[0, 100],
        ticksuffix="%",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    )
)

# endpoint annotation
fig.add_annotation(
    x=x.iloc[-1],
    y=y.iloc[-1],
    text=f"{y.iloc[-1]:.1f}%",
    showarrow=False,
    xshift=20,
    font=dict(
        color="#f7dfb3",
        size=14
    )
)

fig.show()

In [92]:
# save fig 
fig.write_html(f"../website/charts/percent_food_expenditure.html", include_plotlyjs="cdn", full_html=False)

## Food CPI vs General CPI 

Food cpi vs general cpi  = whether food prices have inflated faster or slower than everything else in the economy over time / is food getting more expensive relative to other things Americans buy?

In [6]:
# import data 

food_cpi = pd.read_csv("../data/total/CPIUFDSL.csv", low_memory=False)
general_cpi = pd.read_csv("../data/total/CPIAUCSL.csv", low_memory=False)

In [7]:
# clean data 

# filter to 1960 and later 
food_cpi_clean = food_cpi.loc[food_cpi['observation_date']>='1960-01-01']
general_cpi_clean = general_cpi.loc[general_cpi['observation_date']>='1960-01-01']

# rename 
food_cpi_clean = food_cpi_clean.rename(columns={'CPIUFDSL':'food_cpi'})
general_cpi_clean = general_cpi_clean.rename(columns={'CPIAUCSL':'total_cpi'})

# merge 
both_cpi = food_cpi_clean.merge(general_cpi_clean, how='outer')

# reindex to January 1960 = 100
base_food = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "food_cpi"].values
base_all = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "total_cpi"].values

both_cpi['food_cpi_reindex'] = (both_cpi['food_cpi'] / base_food) * 100
both_cpi['total_cpi_reindex'] = (both_cpi['total_cpi'] / base_all) * 100

# drop na 
both_cpi = both_cpi.dropna()

In [8]:
# line chart of food as a percent of personal consumption expenditures

# data 
x = both_cpi["observation_date"]
food_cpi = both_cpi["food_cpi_reindex"]
total_cpi = both_cpi["total_cpi_reindex"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=total_cpi,
        mode="lines",
        name="All Items",
        line=dict(
            color="#458dbd",
            width=4
        ),
        hovertemplate=
        "All Items CPI: %{y:.1f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_cpi,
        mode="lines",
        name="Food",
        line=dict(
            color='#67a384',
            width=4
        ),
        hovertemplate=
        "Food CPI: %{y:.1f}<extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=total_cpi,
        mode="lines",
        name="All Items",
        line=dict(
            color="#458dbd",
            width=4
        ),
        fill='tonexty',
        fillcolor='rgba(214, 162, 58, 0.3)',
        hoverinfo='skip',
        showlegend=False
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Food vs. Overall Price Inflation",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Consumer Price Index for All Urban Customers",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.15,
        xanchor="right",
        x=1,
        itemclick=False,
        itemdoubleclick=False
    )
)

fig.show()

In [133]:
# save fig 
fig.write_html(f"../website/charts/food_vs_overall_inflation.html", include_plotlyjs="cdn", full_html=False)

## Food at Home vs Food Away from Home

Food at home vs food away from home CPIs over time = whether eating at home vs eating out was cheaper at certain times 

In [9]:
# import data 

food_home_cpi = pd.read_csv("../data/categories/CUUR0000SAF11.csv", low_memory=False)
food_away_cpi = pd.read_csv("../data/categories/CUUR0000SEFV.csv", low_memory=False)

In [10]:
# clean data 

# filter to 1960 and later 
food_home_cpi_clean = food_home_cpi.loc[food_home_cpi['observation_date']>='1960-01-01']
food_away_cpi_clean = food_away_cpi.loc[food_away_cpi['observation_date']>='1960-01-01']

# rename 
food_home_cpi_clean = food_home_cpi_clean.rename(columns={'CUUR0000SAF11':'food_home_cpi'})
food_away_cpi_clean = food_away_cpi_clean.rename(columns={'CUUR0000SEFV':'food_away_cpi'})

# merge 
both_cpi = food_home_cpi_clean.merge(food_away_cpi_clean, how='outer')

# reindex to January 1960 = 100
base_home = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "food_home_cpi"].values
base_away = both_cpi.loc[both_cpi['observation_date']=="1960-01-01", "food_away_cpi"].values

both_cpi['food_home_cpi_reindex'] = (both_cpi['food_home_cpi'] / base_home) * 100
both_cpi['food_away_cpi_reindex'] = (both_cpi['food_away_cpi'] / base_away) * 100

# drop na 
both_cpi = both_cpi.dropna()

In [11]:
# line chart of food at home vs food away from home cpis 

# data 
x = both_cpi["observation_date"]
food_home_cpi = both_cpi["food_home_cpi_reindex"]
food_away_cpi = both_cpi["food_away_cpi_reindex"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_home_cpi,
        mode="lines",
        name="Food at Home",
        line=dict(
            color='#0c629b',
            width=4
        ),
        hovertemplate=
        "Food at Home CPI: %{y:.1f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_away_cpi,
        mode="lines",
        name="Food Away from Home",
        line=dict(
            color="#22794c",
            width=4
        ),
        hovertemplate=
        "Food Away from Home CPI: %{y:.1f}<extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=food_home_cpi,
        mode="lines",
        name="Food at Home",
        line=dict(
            color='#0c629b',
            width=4
        ),
        fill='tonexty',
        fillcolor='rgba(214, 162, 58, 0.3)',
        hoverinfo='skip',
        showlegend=False
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Food at Home vs. Away from Home Price Inflation",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Consumer Price Index for All Urban Customers",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.15,
        xanchor="right",
        x=1,
        itemclick=False,
        itemdoubleclick=False
    )
)

fig.show()

In [12]:
# save fig 
fig.write_html(f"../website/charts/home_vs_away.html", include_plotlyjs="cdn", full_html=False)

## Grocery Category Comparison 

Grocery categories cpi comparison = relative price trajectory of different food types, is cheaper food less healthy now


In [13]:
# import data 

cereal_bakery = pd.read_csv("../data/categories/CUUR0000SAF111.csv", low_memory=False)
meat_fish_eggs = pd.read_csv("../data/categories/CUUR0000SAF112.csv", low_memory=False)
fruit_veg = pd.read_csv("../data/categories/CUUR0000SAF113.csv", low_memory=False)
beverage = pd.read_csv("../data/categories/CUUR0000SAF114.csv", low_memory=False)

# clean data 

# filter to 1960 and later 
cereal_bakery = cereal_bakery.loc[cereal_bakery['observation_date']>='1967-01-01']
meat_fish_eggs = meat_fish_eggs.loc[meat_fish_eggs['observation_date']>='1967-01-01']
fruit_veg = fruit_veg.loc[fruit_veg['observation_date']>='1967-01-01']
beverage = beverage.loc[beverage['observation_date']>='1967-01-01']

# rename 
cereal_bakery = cereal_bakery.rename(columns={'CUUR0000SAF111':'cereal_bakery_cpi'})
meat_fish_eggs = meat_fish_eggs.rename(columns={'CUUR0000SAF112':'meat_fish_eggs_cpi'})
fruit_veg = fruit_veg.rename(columns={'CUUR0000SAF113':'fruit_veg_cpi'})
beverage = beverage.rename(columns={'CUUR0000SAF114':'beverage_cpi'})

# reindex to January 1967 = 100
base_cb = cereal_bakery.loc[cereal_bakery['observation_date']=="1967-01-01", "cereal_bakery_cpi"].values
base_mfe = meat_fish_eggs.loc[meat_fish_eggs['observation_date']=="1967-01-01", "meat_fish_eggs_cpi"].values
base_fv = fruit_veg.loc[fruit_veg['observation_date']=="1967-01-01", "fruit_veg_cpi"].values
base_b = beverage.loc[beverage['observation_date']=="1967-01-01", "beverage_cpi"].values

cereal_bakery['cereal_bakery_cpi_reindex'] = (cereal_bakery['cereal_bakery_cpi'] / base_cb) * 100
meat_fish_eggs['meat_fish_eggs_cpi_reindex'] = (meat_fish_eggs['meat_fish_eggs_cpi'] / base_mfe) * 100
fruit_veg['fruit_veg_cpi_reindex'] = (fruit_veg['fruit_veg_cpi'] / base_fv) * 100
beverage['beverage_cpi_reindex'] = (beverage['beverage_cpi'] / base_b) * 100

# drop na 
cereal_bakery = cereal_bakery.dropna()
meat_fish_eggs = meat_fish_eggs.dropna()
fruit_veg = fruit_veg.dropna()
beverage = beverage.dropna()

In [14]:
# line chart of grocery category cpis 

# data 
x = cereal_bakery["observation_date"]
cb = cereal_bakery["cereal_bakery_cpi_reindex"]
mfe = meat_fish_eggs["meat_fish_eggs_cpi_reindex"]
fv = fruit_veg["fruit_veg_cpi_reindex"]
b = beverage["beverage_cpi_reindex"]

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=cb,
        mode="lines",
        name="Cereal and Bakery",
        line=dict(
            color="#c48b22",
            width=4
        ),
        hovertemplate=
        "Cereal and Bakery CPI: %{y:.1f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=mfe,
        mode="lines",
        name="Meat, Poultry, Fish, Eggs",
        line=dict(
            color="#b84141",
            width=4
        ),
        hovertemplate=
        "Meat, Poultry, Fish, Eggs CPI: %{y:.1f}<extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=fv,
        mode="lines",
        name="Fruit and Vegetables",
        line=dict(
            color="#22794c",
            width=4
        ),
        hovertemplate=
        "Fruit and Vegetables CPI: %{y:.1f}<extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=b,
        mode="lines",
        name="Beverages",
        line=dict(
            color='#0c629b',
            width=4
        ),
        hovertemplate=
        "Beverages CPI: %{y:.1f}<extra></extra>"
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Grocery Price Inflation by Category",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=13
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Consumer Price Index for All Urban Customers",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        yanchor="top",
        y=1.05,
        xanchor="right",
        x=1
    )
)

fig.show()

In [198]:
# save fig 
fig.write_html(f"../website/charts/grocery_categories.html", include_plotlyjs="cdn", full_html=False)

## Animated Bar Chart by Outlet 

Monthly sales by outlet = compare sales by outlet over time in order to see whether more people are food at grocery stores, large stores like costco, restaurants, etc

In [15]:
# import data 

sales_by_outlet = pd.read_csv("../data/usda/monthly_sales_by_outlet.csv", low_memory=False)

In [25]:
# clean data 

# select columns 
sales_by_outlet_clean = sales_by_outlet[['Year',
                                   'Month',
                                   'Food sales at grocery stores (millions of constant U.S. dollars (1988=100))',
                                   'Food sales at warehouse club and supercenter (millions of constant U.S. dollars (1988=100))',
                                   'Other food-at-home sales (millions of constant U.S. dollars (1988=100))',
                                   'Food sales at full-service restaurants (millions of constant U.S. dollars (1988=100))',
                                   'Food sales at limited-service restaurant (millions of constant U.S. dollars (1988=100))',
                                   'Other food-away-from-home sales (millions of constant U.S. dollars (1988=100))',
                                   'Total food-at-home sales (millions of constant U.S. dollars (1988=100))',
                                   'Total food-away-from-home sales (millions of constant U.S. dollars (1988=100))',
                                   'Total food sales (millions of constant U.S. dollars (1988=100))']]

# rename columns 
sales_by_outlet_clean = sales_by_outlet_clean.rename(columns = {
    'Food sales at grocery stores (millions of constant U.S. dollars (1988=100))': 'grocery_stores',
    'Food sales at warehouse club and supercenter (millions of constant U.S. dollars (1988=100))': 'warehouse_supercenter',
    'Other food-at-home sales (millions of constant U.S. dollars (1988=100))': 'other_food_at_home',
    'Food sales at full-service restaurants (millions of constant U.S. dollars (1988=100))': 'full_restaurants',
    'Food sales at limited-service restaurant (millions of constant U.S. dollars (1988=100))': 'limited_restaurants',
    'Other food-away-from-home sales (millions of constant U.S. dollars (1988=100))': 'other_food_away',
    'Total food-at-home sales (millions of constant U.S. dollars (1988=100))': 'total_food_at_home',
    'Total food-away-from-home sales (millions of constant U.S. dollars (1988=100))': 'total_food_away',
    'Total food sales (millions of constant U.S. dollars (1988=100))': 'total_overall'
})

# combine year and month into date column 
sales_by_outlet_clean['date'] = pd.to_datetime(sales_by_outlet_clean['Month'] + ' ' + sales_by_outlet_clean['Year'].astype(str), format='%B %Y')
sales_by_outlet_clean['Date '] = sales_by_outlet_clean['date'].dt.strftime('%B %Y')

# convert to long format
sales_by_outlet_long = sales_by_outlet_clean.melt(
    id_vars=['Year', 'Month', 'date', 'Date '],
    value_vars=[
        'grocery_stores',
        'warehouse_supercenter',
        'other_food_at_home',
        'full_restaurants',
        'limited_restaurants',
        'other_food_away',
        # 'total_food_at_home',
        # 'total_food_away',
        # 'total_overall'
    ],
    var_name='category',
    value_name='sales'
)

# convert sales to float 
sales_by_outlet_long['sales'] = sales_by_outlet_long['sales'].astype(str).str.replace(',', '', regex=False).astype(float)

# sort values 
sales_by_outlet_long = sales_by_outlet_long.sort_values(by=['date','category'])

# add columns for hover data 
sales_by_outlet_long['Category '] = sales_by_outlet_long['category'].map({
    'grocery_stores': ' Grocery Stores',
    'warehouse_supercenter': ' Warehouse Clubs',
    'other_food_at_home': ' Other, at Home',
    'full_restaurants': ' Full Restaurants',
    'limited_restaurants': ' Limited Restaurants',
    'other_food_away': ' Other, Away from Home',
})
sales_by_outlet_long['Sales '] = ' $'+sales_by_outlet_long['sales'].round(2).astype(str)

In [26]:
sales_by_outlet_long

,Year,Month,date,Date,category,sales,Category,Sales
1343,1997,January,1997-01-01,January 1997,full_restaurants,7180.08,Full Restaurants,$7180.08
335,1997,January,1997-01-01,January 1997,grocery_stores,16266.17,Grocery Stores,$16266.17
1679,1997,January,1997-01-01,January 1997,limited_restaurants,6512.30,Limited Restaurants,$6512.3
1007,1997,January,1997-01-01,January 1997,other_food_at_home,3941.81,"Other, at Home",$3941.81
2015,1997,January,1997-01-01,January 1997,other_food_away,3107.07,"Other, Away from Home",$3107.07
...,...,...,...,...,...,...,...,...
0,2024,December,2024-12-01,December 2024,grocery_stores,20233.37,Grocery Stores,$20233.37
1344,2024,December,2024-12-01,December 2024,limited_restaurants,14623.75,Limited Restaurants,$14623.75
672,2024,December,2024-12-01,December 2024,other_food_at_home,6606.61,"Other, at Home",$6606.61
1680,2024,December,2024-12-01,December 2024,other_food_away,7411.56,"Other, Away from Home",$7411.56


In [29]:
# make plot 

# colors
color_map = {
    'grocery_stores': "#458dbd",
    'warehouse_supercenter': "#67a384",
    'other_food_at_home': "#adddfc",
    'full_restaurants': "#b84141",
    'limited_restaurants': "#e38a72",
    'other_food_away': "#e3b96b",
}

# plot 
fig = px.bar(sales_by_outlet_long, 
             x="category", 
             y="sales", 
             color="category",
             color_discrete_map=color_map,
             animation_frame="date",
             animation_group="category",
             range_y=[0,25000],
             hover_name="Date ",
             hover_data={
                "Category ":True,
                "Sales ":True,
                "category":False,
                "date":False,
                "sales":False
                },
             )

fig.update_traces(
    marker_line_width=0,
)


fig.update_layout(
    # title
    title={
        'text': 'Food Sales by Outlet Type over Time',
        'x': 0.02,
        'xanchor': 'left',
        'font': {
            'size': 24,
            'family': 'Playfair Display',
            'color': '#f5f5f5'
        }
    },

    # plot background
    plot_bgcolor='#1a1a1a',

    # entire figure background
    paper_bgcolor='#1a1a1a',

    # x-axis
    xaxis=dict(
        title='Outlet Category',
        title_font=dict(
            size=17,
            family="Source Sans 3",
            color='#f5f5f5'
        ),
        tickfont=dict(
            size=12,
            color='#f5f5f5'
        )
    ),

    # y-axis
    yaxis=dict(
        title='Millions of constant U.S. dollars<br>(1988=100)',
        title_font=dict(
            size=17,
            family="Source Sans 3",
            color='#f5f5f5'
        ),
        tickfont=dict(
            size=16,
            color='#f5f5f5'
        )
    ),

    # legend
    showlegend=False
)

fig.update_xaxes(
    tickvals=[
        'grocery_stores',
        'warehouse_supercenter',
        'other_food_at_home',
        'full_restaurants',
        'limited_restaurants',
        'other_food_away'
    ],
    ticktext=[
        'Grocery Stores',
        'Warehouse Clubs',
        'Other, at Home',
        'Full Restaurants',
        'Limited Restaurants',
        'Other, Away from Home'
    ]
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.08)",
    zerolinecolor="rgba(255,255,255,0.08)",
)

for step in fig.layout.sliders[0].steps:
    step["label"] = step["label"][:7]

fig.update_layout(

    # animation slider
    sliders=[{
        'currentvalue': {
            'font': {'color': '#f5f5f5'},
            'prefix': 'Date: ',
            'visible': True,
        },

        # tick labels on slider
        'tickcolor': "rgba(245, 245, 245, 0.2)",

        # slider styling
        'bgcolor': '#1a1a1a',
        'bordercolor': "rgba(245, 245, 245, 0.5)",
        'borderwidth': 1,

        # font for slider step labels
        'font': {
            'color': "rgba(245, 245, 245, 0.5)",
        }
    }],

    # buttons
    updatemenus=[{
        'type': 'buttons',

        'font': {
            'color': "rgba(245, 245, 245, 0.5)",
        },

        'bgcolor': '#1a1a1a',
        'bordercolor': "rgba(245, 245, 245, 0.5)",

    }]
)

fig.show()

In [30]:
# save fig 
fig.write_html(f"../website/charts/spending_by_outlet.html", include_plotlyjs="cdn", full_html=False)

In [18]:
# line chart of grocery category cpis 

# data 
x = sales_by_outlet_clean["date"]
sales_by_outlet_clean['total_food_at_home'] = sales_by_outlet_clean['total_food_at_home'].astype(str).str.replace(',', '', regex=False).astype(float)
sales_by_outlet_clean['total_food_away'] = sales_by_outlet_clean['total_food_away'].astype(str).str.replace(',', '', regex=False).astype(float)
sales_by_outlet_clean['total_overall'] = sales_by_outlet_clean['total_overall'].astype(str).str.replace(',', '', regex=False).astype(float)
total_home_percent = sales_by_outlet_clean['total_food_at_home']/sales_by_outlet_clean['total_overall']
total_away_percent = sales_by_outlet_clean['total_food_away']/sales_by_outlet_clean['total_overall']

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=total_home_percent,
        mode="lines",
        name="Home",
        line=dict(
            color="#c48b22",
            width=4
        ),
        hovertemplate=
        "Home: %{y:.3f}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=total_away_percent,
        mode="lines",
        name="Away",
        line=dict(
            color="#b84141",
            width=4
        ),
        hovertemplate=
        "Away: %{y:.3f}<extra></extra>"
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Grocery Price Inflation by Category",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=13
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Consumer Price Index for All Urban Customers",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        yanchor="top",
        y=0.15,
        xanchor="right",
        x=1
    )
)

fig.show()

## Inequality 1984 vs 2024

DOUBLE BAR CHART w/ 1984 vs 2024 for all 3 percentiles 
LINE CHART for total percentage over time and percentiles

Food at home expenditures vs income, compare across percentiles over time 

pre-tax income 
- slightly understates food spend percentage for lower-income households bc income is higher than what they actually have 
- undercounts their actual resources (SNAP benefits, earned income tax credit, etc)
- the most consistently reported figure

In [19]:
# import data 

all_food_spend = pd.read_csv("../data/quintiles/CXUFOODHOMELB0101M.csv", low_memory=False)
bottom_food_spend = pd.read_csv("../data/quintiles/CXUFOODHOMELB0102M.csv", low_memory=False)
middle_food_spend = pd.read_csv("../data/quintiles/CXUFOODHOMELB0104M.csv", low_memory=False)
top_food_spend = pd.read_csv("../data/quintiles/CXUFOODHOMELB0106M.csv", low_memory=False)
all_income = pd.read_csv("../data/quintiles/CXUINCBEFTXLB0101M.csv", low_memory=False)
bottom_income = pd.read_csv("../data/quintiles/CXUINCBEFTXLB0102M.csv", low_memory=False)
middle_income = pd.read_csv("../data/quintiles/CXUINCBEFTXLB0104M.csv", low_memory=False)
top_income = pd.read_csv("../data/quintiles/CXUINCBEFTXLB0106M.csv", low_memory=False)

# clean data 

# rename 
all_food_spend = all_food_spend.rename(columns={'CXUFOODHOMELB0101M':'all_food_spend'})
bottom_food_spend = bottom_food_spend.rename(columns={'CXUFOODHOMELB0102M':'bottom_food_spend'})
middle_food_spend = middle_food_spend.rename(columns={'CXUFOODHOMELB0104M':'middle_food_spend'})
top_food_spend = top_food_spend.rename(columns={'CXUFOODHOMELB0106M':'top_food_spend'})
all_income = all_income.rename(columns={'CXUINCBEFTXLB0101M':'all_income'})
bottom_income = bottom_income.rename(columns={'CXUINCBEFTXLB0102M':'bottom_income'})
middle_income = middle_income.rename(columns={'CXUINCBEFTXLB0104M':'middle_income'})
top_income = top_income.rename(columns={'CXUINCBEFTXLB0106M':'top_income'})

# drop na 
all_food_spend = all_food_spend.dropna()
bottom_food_spend = bottom_food_spend.dropna()
middle_food_spend = middle_food_spend.dropna()
top_food_spend = top_food_spend.dropna()
all_income = all_income.dropna()
bottom_income = bottom_income.dropna()
middle_income = middle_income.dropna()
top_income = top_income.dropna()

In [20]:
## line chart of food spending percentage by income percentile 

# data 
x = all_food_spend["observation_date"]
overall = (all_food_spend["all_food_spend"]/all_income['all_income']) *100
bottom = (bottom_food_spend["bottom_food_spend"]/bottom_income['bottom_income']) *100
middle = (middle_food_spend["middle_food_spend"]/middle_income['middle_income']) *100
top = (top_food_spend["top_food_spend"]/top_income['top_income']) *100

# figure
fig = go.Figure()

# line plot 
fig.add_trace(
    go.Scatter(
        x=x,
        y=overall,
        mode="lines",
        name="Overall",
        line=dict(
            color="#c48b22",
            width=4
        ),
        hovertemplate=
        "Overall: %{y:.2f}%<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=bottom,
        mode="lines",
        name="Bottom 20%",
        line=dict(
            color="#b84141",
            width=4
        ),
        hovertemplate=
        "Bottom 20%: %{y:.1f}%<extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=middle,
        mode="lines",
        name="Middle 20%",
        line=dict(
            color="#22794c",
            width=4
        ),
        hovertemplate=
        "Middle 20%: %{y:.1f}%<extra></extra>"
    )
)
fig.add_trace(
    go.Scatter(
        x=x,
        y=top,
        mode="lines",
        name="Top 20%",
        line=dict(
            color='#0c629b',
            width=4
        ),
        hovertemplate=
        "Top 20%: %{y:.1f}%<extra></extra>"
    )
)

# title and layout 
fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Share of Income Spent on Groceries Over Time",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=13
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    hovermode="x unified",

    xaxis=dict(
        title="Date",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Food at Home as % of Income Before Taxes",
        tickfont=dict(size=14),
        ticksuffix="%",
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False
    ),

    legend=dict(
        orientation="h",
        yanchor="top",
        y=1.05,
        xanchor="right",
        x=1
    )
)

fig.show()

In [232]:
# save fig 
fig.write_html(f"../website/charts/line_grocery_income.html", include_plotlyjs="cdn", full_html=False)

In [21]:
## compare food spending between 2 years across incomes 

years_to_compare = [1984, 2024]
quintiles = ["Bottom 20%", "Middle 20%", "Top 20%"]

# get values for each category 
ratios = {"Bottom 20%": bottom,
          "Middle 20%": middle,
          "Top 20%": top}
values_1984 = [ratios[q][0] for q in quintiles]
values_2024 = [ratios[q].iloc[-1] for q in quintiles]

# plot
fig = go.Figure()

fig.add_trace(go.Bar(
    name="1984",
    x=quintiles,
    y=values_1984,
    marker_color="#0c629b",
    hovertemplate="1984<br>%{x}<br>%{y}<extra></extra>"
))

fig.add_trace(go.Bar(
    name="2024",
    x=quintiles,
    y=values_2024,
    marker_color='#22794c',
    hovertemplate="2024<br>%{x}<br>%{y}<extra></extra>"
))

fig.update_layout(
    template="plotly_dark",

    title=dict(
        text="Share of Income Spent on Groceries by Income Group",
        x=0.02,
        xanchor="left",
        font=dict(
            family="Playfair Display",
            size=24
        )
    ),

    font=dict(
        family="Source Sans 3",
        color="#f5f0e8",
        size=14
    ),

    paper_bgcolor="#1a1a1a",
    plot_bgcolor="#1a1a1a",

    margin=dict(l=60, r=40, t=80, b=60),

    barmode="group",
    xaxis=dict(
        title="Income Quintile",
        showgrid=False,
        tickfont=dict(size=14),
        zeroline=False,
        ticklabelstandoff=5
    ),

    yaxis=dict(
        title="Food at Home as % of Income Before Taxes",
        tickfont=dict(size=14),
        gridcolor="rgba(255,255,255,0.08)",
        zeroline=False,
        ticksuffix="%",
        range=[0, max(values_1984 + values_2024) * 1.2]
    ),
   
    legend=dict(
        orientation="h", 
        yanchor="bottom", 
        y=1.02, 
        xanchor="right", 
        x=1
    ),

    bargap=0.1,
    bargroupgap=0
)

fig.show()

In [229]:
# save fig 
fig.write_html(f"../website/charts/bar_grocery_income.html", include_plotlyjs="cdn", full_html=False)

## Per State 

State sales per capita (udsa) / per capita personal income per state = food burden by state over time

In [22]:
# import data 

state_food = pd.read_csv("../data/usda/state_sales_per_capita_no_taxes_tips.csv", low_memory=False)

folder_path = "../data/states"
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
dfs = []
for file in csv_files:
    df = pd.read_csv(file)
    df['state_abbr'] = file[15:17]
    col_name = file[15:21]
    df = df.rename(columns={col_name:'per_capita_personal_income'})
    dfs.append(df)

state_personal_income = pd.concat(
    dfs,
    ignore_index=True
)

# clean data 

# state mapping
state_map = {
    "AL": "Alabama",
    "AK": "Alaska",
    "AZ": "Arizona",
    "AR": "Arkansas",
    "CA": "California",
    "CO": "Colorado",
    "CT": "Connecticut",
    "DE": "Delaware",
    "FL": "Florida",
    "GA": "Georgia",
    "HI": "Hawaii",
    "ID": "Idaho",
    "IL": "Illinois",
    "IN": "Indiana",
    "IA": "Iowa",
    "KS": "Kansas",
    "KY": "Kentucky",
    "LA": "Louisiana",
    "ME": "Maine",
    "MD": "Maryland",
    "MA": "Massachusetts",
    "MI": "Michigan",
    "MN": "Minnesota",
    "MS": "Mississippi",
    "MO": "Missouri",
    "MT": "Montana",
    "NE": "Nebraska",
    "NV": "Nevada",
    "NH": "New Hampshire",
    "NJ": "New Jersey",
    "NM": "New Mexico",
    "NY": "New York",
    "NC": "North Carolina",
    "ND": "North Dakota",
    "OH": "Ohio",
    "OK": "Oklahoma",
    "OR": "Oregon",
    "PA": "Pennsylvania",
    "RI": "Rhode Island",
    "SC": "South Carolina",
    "SD": "South Dakota",
    "TN": "Tennessee",
    "TX": "Texas",
    "UT": "Utah",
    "VT": "Vermont",
    "VA": "Virginia",
    "WA": "Washington",
    "WV": "West Virginia",
    "WI": "Wisconsin",
    "WY": "Wyoming",
    "DC": "District of Columbia"
}

# filter to 1997 and later
state_personal_income = state_personal_income.loc[state_personal_income['observation_date']>='1997-01-01']

# add year and state cols
state_personal_income['year'] = pd.to_datetime(state_personal_income['observation_date']).dt.year
state_personal_income['state'] = state_personal_income["state_abbr"].map(state_map)

# select columns 
state_food = state_food[['Year', 'State',
       'Food-at-home per capita sales without taxes and tips (nominal U.S. dollars)',
       'Food-away-from-home per capita sales without taxes and tips (nominal U.S. dollars)',
       'Total food per capita sales without taxes and tips (nominal U.S. dollars)']]

# rename columns 
state_food = state_food.rename(columns = {
    'Year':'year', 
    'State':'state',
    'Food-at-home per capita sales without taxes and tips (nominal U.S. dollars)':'food_home_pc_sales',
    'Food-away-from-home per capita sales without taxes and tips (nominal U.S. dollars)':'food_away_pc_sales',
    'Total food per capita sales without taxes and tips (nominal U.S. dollars)':'total_food_pc_sales'
})

# convert sales to float 
state_food['food_home_pc_sales'] = state_food['food_home_pc_sales'].astype(str).str.replace(',', '', regex=False).astype(float)
state_food['food_away_pc_sales'] = state_food['food_away_pc_sales'].astype(str).str.replace(',', '', regex=False).astype(float)
state_food['total_food_pc_sales'] = state_food['total_food_pc_sales'].astype(str).str.replace(',', '', regex=False).astype(float)

# sort values 
state_food = state_food.sort_values(by=['state', 'year'])

# merge
state_food_vs_income = state_food.merge(state_personal_income, on=['year','state'], how='inner')

# add percentage columns 
state_food_vs_income['total_food_income_percentage'] = (state_food_vs_income['total_food_pc_sales']/state_food_vs_income['per_capita_personal_income']) *100
state_food_vs_income['home_food_income_percentage'] = (state_food_vs_income['food_home_pc_sales']/state_food_vs_income['per_capita_personal_income']) *100
state_food_vs_income['away_food_income_percentage'] = (state_food_vs_income['food_away_pc_sales']/state_food_vs_income['per_capita_personal_income']) *100

# add columns for clean hover data 
state_food_vs_income['Year '] = ' '+state_food_vs_income['year'].astype(str)
state_food_vs_income['Percentage '] = ' '+state_food_vs_income['total_food_income_percentage'].round(2).astype(str) + "%"

In [291]:
display(state_food_vs_income)

,year,state,food_home_pc_sales,food_away_pc_sales,total_food_pc_sales,observation_date,per_capita_personal_income,state_abbr,total_food_income_percentage,home_food_income_percentage,away_food_income_percentage,Year,Percentage
0,1997,Alabama,1237.56,860.34,2097.90,1997-01-01,21487,AL,9.763578,5.759576,4.004002,1997,9.76%
1,1998,Alabama,1263.70,923.23,2186.93,1998-01-01,22651,AL,9.654894,5.579003,4.075891,1998,9.65%
2,1999,Alabama,1312.83,973.54,2286.37,1999-01-01,23296,AL,9.814432,5.635431,4.179001,1999,9.81%
3,2000,Alabama,1333.11,995.83,2328.94,2000-01-01,24258,AL,9.600709,5.495548,4.105161,2000,9.6%
4,2001,Alabama,1376.06,1043.00,2419.06,2001-01-01,25046,AL,9.658468,5.494131,4.164338,2001,9.66%
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1423,2020,Wyoming,2942.87,2573.97,5516.84,2020-01-01,65094,WY,8.475190,4.520954,3.954235,2020,8.48%
1424,2021,Wyoming,3133.39,3274.71,6408.10,2021-01-01,70962,WY,9.030326,4.415589,4.614737,2021,9.03%
1425,2022,Wyoming,3552.79,3520.49,7073.28,2022-01-01,76394,WY,9.258947,4.650614,4.608333,2022,9.26%
1426,2023,Wyoming,3588.91,3656.43,7245.34,2023-01-01,81918,WY,8.844625,4.381101,4.463525,2023,8.84%


In [ ]:
# color
color_continuous_scale=['#1a5c3a',"#67a384", "#c48b22"]

# plot
fig = px.choropleth(
    state_food_vs_income,
    locations="state_abbr",
    locationmode="USA-states",
    color="total_food_income_percentage",
    hover_name="state",
    hover_data={
        "total_food_income_percentage": False,
        "year": False,
        'state_abbr':False,
        'Year ':True, 
        'Percentage ': True
    },
    animation_frame="year",
    scope="usa",
    color_continuous_scale=color_continuous_scale,
    range_color=(
        state_food_vs_income["total_food_income_percentage"].min(),
        state_food_vs_income["total_food_income_percentage"].max()
    ),
)

fig.update_layout(

    # title
    title={
        'text': "Food Sales as % of Personal Income by State Over Time (Per Capita)",
        'x': 0.02,
        'xanchor': 'left',
        'font': {
            'size': 24,
            'family': 'Playfair Display',
            'color': '#f5f0e8'
        }
    },

    # plot background
    plot_bgcolor='#1a1a1a',

    # entire figure background
    paper_bgcolor='#1a1a1a',

)

# map background
fig.update_geos(
    bgcolor='#1a1a1a',
    lakecolor='#f5f0e8',
)
fig.update_traces(
    marker_line_color="rgba(245, 245, 245, 0.5)",
    marker_line_width=0.5
)

# animation styling 
fig.update_layout(

    # animation slider
    sliders=[{
        'currentvalue': {
            'font': {'color': '#f5f0e8'},
            'prefix': 'Year: ',
            'visible': True,
        },

        # tick labels on slider
        'tickcolor': "rgba(245, 245, 245, 0.2)",

        # slider styling
        'bgcolor': '#1a1a1a',
        'bordercolor': "rgba(245, 245, 245, 0.5)",
        'borderwidth': 1,

        # font for slider step labels
        'font': {
            'color': "rgba(245, 245, 245, 0.5)",
        }
    }],

    # buttons
    updatemenus=[{
        'type': 'buttons',

        'font': {
            'color': "rgba(245, 245, 245, 0.5)",
        },

        'bgcolor': '#1a1a1a',
        'bordercolor': "rgba(245, 245, 245, 0.5)",

    }]
)

fig.update_coloraxes(
    colorbar_title="Percentage",
    colorbar_ticksuffix="%",
    colorbar_tickfont=dict(
        color="#f5f0e8",
        size=12
    ),
    colorbar_title_font=dict(
        color="#f5f0e8",
        size=14
    )
)

fig.show()

In [40]:
# save fig 
fig.write_html(f"../website/charts/map_food_sales_income.html", include_plotlyjs="cdn", full_html=False)